# Multi-Frame Blind Deconvolution with Linear Equality Constraints — Implementation / 구현

**Paper**: Löfdahl, M. G. (2002). *Multi-frame blind deconvolution with linear equality constraints.* Proc. SPIE 4792-21. DOI: 10.1117/12.451791.

**English.** This notebook illustrates the building blocks of the MFBD–LEC framework:
1. Zernike polynomial basis on the unit disk (wavefront representation).
2. Random atmospheric phase screens via Kolmogorov-weighted Zernike coefficients.
3. Point-spread function (PSF) from a generalized pupil: `h = |IFT{A·exp(iφ)}|²`.
4. Forward model: object convolved with a time-varying PSF plus Gaussian noise.
5. Single-frame Wiener deconvolution (and why it fails in the blind case).
6. Multi-frame averaging and the Wiener-filtered object estimate analogous to Löfdahl's Eq. (4).

**Korean.** 이 노트북은 MFBD–LEC 프레임워크의 구성 요소를 예시한다:
1. 단위 원판 상의 Zernike 다항식 기저(파면 표현).
2. Kolmogorov 가중 Zernike 계수로 만든 무작위 대기 위상 스크린.
3. 일반화된 동공에서의 점 확산 함수(PSF): `h = |IFT{A·exp(iφ)}|²`.
4. 전방 모델: 물체와 시간 가변 PSF의 합성곱 + 가우시안 잡음.
5. 단일 프레임 Wiener 디컨볼루션(그리고 블라인드 경우의 실패).
6. 다중 프레임 평균과 Löfdahl 식 (4)에 상응하는 Wiener 필터 물체 추정.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from numpy.fft import fft2, ifft2, fftshift, ifftshift

plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 11

rng = np.random.default_rng(seed=42)

## Part 1: Zernike Polynomial Basis / Zernike 다항식 기저

**English.** Zernike polynomials $Z_n^m(\rho,\theta)$ form an orthogonal basis on the unit disk, ideal for representing optical aberrations. We use the OSA/ANSI single-index convention: `j = n(n+2)/2 + m/2` approximation; simpler, we generate them by (n,m) and order lexicographically.

**Korean.** Zernike 다항식 $Z_n^m(\rho,\theta)$는 단위 원판 상의 직교 기저로, 광학 수차 표현에 이상적이다. 여기서는 (n,m)으로 생성한 뒤 사전식으로 정렬한다.

In [ ]:
from math import factorial


def zernike_radial(n: int, m: int, rho: np.ndarray) -> np.ndarray:
    """Compute the radial part R_n^m(rho) of a Zernike polynomial.

    Args:
        n: Radial order (non-negative integer).
        m: Azimuthal order, |m| <= n and (n - |m|) must be even.
        rho: Normalized radial coordinate in [0, 1].

    Returns:
        Radial polynomial values with the same shape as rho.
    """
    m = abs(m)
    if (n - m) % 2 != 0:
        return np.zeros_like(rho)
    result = np.zeros_like(rho)
    for k in range((n - m) // 2 + 1):
        coeff = (-1) ** k * factorial(n - k)
        coeff /= (
            factorial(k)
            * factorial((n + m) // 2 - k)
            * factorial((n - m) // 2 - k)
        )
        result += coeff * rho ** (n - 2 * k)
    return result


def zernike(n: int, m: int, rho: np.ndarray, theta: np.ndarray) -> np.ndarray:
    """Evaluate a single Zernike polynomial Z_n^m on a polar grid.

    Args:
        n: Radial order.
        m: Azimuthal order (can be negative, zero, positive).
        rho: Normalized radial coordinate array.
        theta: Azimuthal coordinate array.

    Returns:
        Zernike polynomial values with the same shape as rho.
    """
    R = zernike_radial(n, m, rho)
    if m > 0:
        return R * np.cos(m * theta)
    if m < 0:
        return R * np.sin(abs(m) * theta)
    return R


def noll_to_nm(j: int) -> tuple[int, int]:
    """Convert Noll single-index j (>=1) to (n, m) pair.

    Args:
        j: Noll 1-based index (1 = piston, 2 = tip, 3 = tilt, 4 = defocus, ...).

    Returns:
        Tuple (n, m) with n = radial order, m = azimuthal order.
    """
    n = 0
    j1 = j - 1
    while j1 > n:
        n += 1
        j1 -= n
    m = (-1) ** j * ((n % 2) + 2 * ((j1 + ((n + 1) % 2)) // 2))
    return n, m

In [ ]:
# Visualize the first few Zernike modes on a 128x128 grid.
N = 128
y, x = np.indices((N, N)) - N / 2
rho = np.sqrt(x ** 2 + y ** 2) / (N / 2)
theta = np.arctan2(y, x)
pupil_mask = (rho <= 1.0).astype(np.float64)

fig, axes = plt.subplots(3, 5, figsize=(14, 8))
labels = ['piston', 'tip', 'tilt', 'defocus', 'astig-0',
          'astig-45', 'coma-y', 'coma-x', 'trefoil-y', 'trefoil-x',
          'spherical', 'Z12', 'Z13', 'Z14', 'Z15']
for ax, j_noll, lbl in zip(axes.flat, range(1, 16), labels):
    n, m = noll_to_nm(j_noll)
    z = zernike(n, m, rho, theta) * pupil_mask
    ax.imshow(z, cmap='RdBu_r', extent=(-1, 1, -1, 1))
    ax.set_title(f'j={j_noll} (n={n},m={m})\n{lbl}')
    ax.axis('off')
plt.tight_layout()
plt.suptitle('First 15 Zernike modes / 처음 15개 Zernike 모드', y=1.02)
plt.show()

## Part 2: Kolmogorov Atmospheric Phase Screen / Kolmogorov 대기 위상 스크린

**English.** Noll (1976) showed that when the wavefront phase from Kolmogorov turbulence is expanded in Zernike modes, the variance of the $j$-th mode scales approximately as
$$\sigma_j^2 \sim (D/r_0)^{5/3}\cdot c_j, \qquad c_j \propto j^{-11/6}\ \text{for large } j.$$
where $D$ is the pupil diameter and $r_0$ is the Fried parameter. We draw random Zernike coefficients with these variances to create a plausible atmospheric phase screen. Karhunen–Loève modes (Löfdahl's $\psi_m$) are a better basis but low-order Zernikes are close enough for illustration.

**Korean.** Noll(1976)은 Kolmogorov 난류의 파면 위상을 Zernike 모드로 전개하면 $j$번째 모드의 분산이 $\sigma_j^2 \sim (D/r_0)^{5/3}\cdot c_j$로 스케일되며($c_j \propto j^{-11/6}$, $j$가 클 때) $D$는 동공 지름, $r_0$는 Fried 모수임을 보였다. 이 분산으로 무작위 Zernike 계수를 뽑아 그럴듯한 대기 위상 스크린을 만든다. Karhunen–Loève 모드(Löfdahl의 $\psi_m$)가 더 적절한 기저이지만 저차 Zernike로도 설명에 충분하다.

In [ ]:
def kolmogorov_variances(n_modes: int, D_over_r0: float = 10.0) -> np.ndarray:
    """Approximate Noll variances for Zernike coefficients under Kolmogorov statistics.

    Args:
        n_modes: Number of Zernike modes (Noll indexing, piston excluded in practice).
        D_over_r0: Ratio of pupil diameter to Fried parameter (seeing strength).

    Returns:
        Variance array of length n_modes. The piston (j=1) variance is set to 0.
    """
    variances = np.zeros(n_modes)
    # Rough approximation after Noll (1976).
    for j in range(2, n_modes + 1):
        variances[j - 1] = (D_over_r0) ** (5 / 3) * 0.3 * j ** (-11 / 6)
    return variances


def random_phase_screen(
    n_modes: int,
    D_over_r0: float,
    rho: np.ndarray,
    theta: np.ndarray,
    pupil_mask: np.ndarray,
    rng: np.random.Generator,
) -> tuple[np.ndarray, np.ndarray]:
    """Draw a random atmospheric phase screen expanded in Zernike modes.

    Args:
        n_modes: Number of Zernike modes (Noll indexing).
        D_over_r0: Seeing strength D/r_0.
        rho: Normalized radial coordinate grid.
        theta: Azimuthal coordinate grid.
        pupil_mask: Binary pupil mask, same shape as rho.
        rng: NumPy random generator.

    Returns:
        phase: 2-D phase array (radians) inside pupil, zero outside.
        coeffs: Zernike coefficient vector actually used.
    """
    variances = kolmogorov_variances(n_modes, D_over_r0)
    coeffs = rng.standard_normal(n_modes) * np.sqrt(variances)
    phase = np.zeros_like(rho)
    for j in range(2, n_modes + 1):  # skip piston
        n, m = noll_to_nm(j)
        phase += coeffs[j - 1] * zernike(n, m, rho, theta)
    return phase * pupil_mask, coeffs

In [ ]:
# Draw a few sample phase screens
fig, axes = plt.subplots(2, 4, figsize=(14, 7))
for ax in axes.flat:
    phase, _ = random_phase_screen(15, 10.0, rho, theta, pupil_mask, rng)
    im = ax.imshow(phase, cmap='RdBu_r')
    ax.set_title(f'RMS = {phase[pupil_mask > 0].std():.2f} rad')
    ax.axis('off')
    plt.colorbar(im, ax=ax, fraction=0.046)
plt.tight_layout()
plt.suptitle('Random Kolmogorov phase screens (15 Zernikes, D/r0=10) / 무작위 Kolmogorov 위상 스크린', y=1.02)
plt.show()

## Part 3: PSF from Wavefront Phase / 파면 위상으로부터 PSF

**English.** Given a generalized pupil $P = A\,e^{i\phi}$, the coherent amplitude in the image plane is $p = \mathcal{F}^{-1}\{P\}$ and the incoherent PSF is
$$h = |p|^2 = |\mathcal{F}^{-1}\{A\,e^{i\phi}\}|^2.$$
The OTF is the Fourier transform of the PSF and equals the normalized autocorrelation of $P$.

**Korean.** 일반화된 동공 $P = A\,e^{i\phi}$가 주어지면 영상면의 결맞음 진폭은 $p = \mathcal{F}^{-1}\{P\}$이고 비결맞음 PSF는 $h = |p|^2$이다. OTF는 PSF의 푸리에 변환이며 $P$의 정규화된 자기상관과 같다.

In [ ]:
def psf_from_phase(phase: np.ndarray, pupil_mask: np.ndarray) -> np.ndarray:
    """Compute the incoherent PSF from a wavefront phase and pupil mask.

    The PSF is centred via fftshift and peak-normalized (sum = 1) for unit flux.

    Args:
        phase: 2-D phase array in radians.
        pupil_mask: Binary pupil amplitude mask, same shape as phase.

    Returns:
        Real-valued, shifted, unit-sum PSF of the same shape.
    """
    pupil = pupil_mask * np.exp(1j * phase)
    amp = fftshift(ifft2(ifftshift(pupil)))
    psf = np.abs(amp) ** 2
    return psf / psf.sum()


def otf_from_phase(phase: np.ndarray, pupil_mask: np.ndarray) -> np.ndarray:
    """Compute the OTF (complex, un-shifted) from the wavefront phase.

    Args:
        phase: 2-D phase array in radians.
        pupil_mask: Binary pupil amplitude mask.

    Returns:
        Complex OTF suitable for multiplication with an FFT'd image.
    """
    psf = psf_from_phase(phase, pupil_mask)
    return fft2(ifftshift(psf))


# Diffraction-limited reference PSF
psf_diff = psf_from_phase(np.zeros_like(rho), pupil_mask)

fig, axes = plt.subplots(1, 4, figsize=(14, 4))
axes[0].imshow(np.log10(psf_diff + 1e-8), cmap='gray')
axes[0].set_title('Diffraction-limited PSF / 회절 한계 PSF')
axes[0].axis('off')
for k in range(1, 4):
    phase, _ = random_phase_screen(21, 8.0, rho, theta, pupil_mask, rng)
    psf = psf_from_phase(phase, pupil_mask)
    axes[k].imshow(np.log10(psf + 1e-8), cmap='gray')
    axes[k].set_title(f'Aberrated PSF #{k} / 수차 있는 PSF')
    axes[k].axis('off')
plt.tight_layout()
plt.show()

## Part 4: Forward Model and Simulated Solar Granulation / 전방 모델과 시뮬레이션된 태양 입상

**English.** We simulate a quiet-Sun granulation field as an isotropic random texture with a von-Kármán-like power spectrum, then convolve each 'frame' with a different aberrated PSF and add Gaussian noise. This is the $D_j = F\cdot S_j + N_j$ data model of Löfdahl Eq. (2).

**Korean.** 정적 태양 입상장을 von-Kármán 류의 전력 스펙트럼을 가진 등방 무작위 텍스처로 시뮬레이션한 뒤, 각 '프레임'을 서로 다른 수차 PSF와 합성곱하고 가우시안 잡음을 더한다. 이는 Löfdahl 식 (2)의 $D_j = F\cdot S_j + N_j$ 데이터 모델이다.

In [ ]:
def synthetic_granulation(
    shape: tuple[int, int],
    scale: float,
    rng: np.random.Generator,
) -> np.ndarray:
    """Generate a synthetic granulation-like image via filtered Gaussian noise.

    Args:
        shape: (height, width) of the output image.
        scale: Characteristic granule size in pixels.
        rng: NumPy random generator.

    Returns:
        2-D normalized intensity image with mean ~1 and contrast ~10%.
    """
    H, W = shape
    noise = rng.standard_normal(shape)
    ky, kx = np.indices(shape) - np.array([[H / 2]], dtype=float)
    kr = np.sqrt((ky - H / 2) ** 2 + (kx[..., 0].reshape(-1, 1) - W / 2) ** 2)
    kr = np.sqrt((np.indices(shape)[0] - H / 2) ** 2 + (np.indices(shape)[1] - W / 2) ** 2)
    filter_fs = np.exp(-(kr * scale / max(H, W)) ** 2)
    ft = fftshift(fft2(noise)) * filter_fs
    img = np.real(ifft2(ifftshift(ft)))
    img = (img - img.mean()) / img.std()
    return 1.0 + 0.1 * img  # ~10% contrast, typical of photospheric granulation


def convolve_fft(img: np.ndarray, psf: np.ndarray) -> np.ndarray:
    """Convolve an image with a centred PSF using FFT.

    Args:
        img: 2-D image.
        psf: 2-D PSF with peak centred at image centre (fftshift'd).

    Returns:
        Convolved image, same shape as input.
    """
    return np.real(ifft2(fft2(img) * fft2(ifftshift(psf))))


# Generate ground truth granulation
truth = synthetic_granulation((N, N), scale=12.0, rng=rng)

plt.figure(figsize=(6, 6))
plt.imshow(truth, cmap='gray')
plt.colorbar()
plt.title('Ground truth granulation F / 진정 입상 F')
plt.axis('off')
plt.show()

In [ ]:
def simulate_frames(
    truth: np.ndarray,
    n_frames: int,
    n_modes: int,
    D_over_r0: float,
    pupil_mask: np.ndarray,
    rho: np.ndarray,
    theta: np.ndarray,
    noise_sigma: float,
    rng: np.random.Generator,
) -> tuple[np.ndarray, list[np.ndarray], list[np.ndarray]]:
    """Simulate a burst of turbulence-degraded frames of the same object.

    Args:
        truth: Ground-truth object image F.
        n_frames: Number of frames J to generate.
        n_modes: Number of Zernike modes per frame.
        D_over_r0: Atmospheric seeing strength.
        pupil_mask: Binary pupil mask.
        rho: Pupil radial coordinate.
        theta: Pupil azimuthal coordinate.
        noise_sigma: Gaussian noise standard deviation.
        rng: NumPy random generator.

    Returns:
        frames: Array of shape (n_frames, H, W) of noisy blurred images.
        phases: List of phase arrays used for each frame (for diagnostics).
        psfs: List of PSFs used for each frame.
    """
    frames = np.empty((n_frames, *truth.shape))
    phases, psfs = [], []
    for j in range(n_frames):
        phase, _ = random_phase_screen(n_modes, D_over_r0, rho, theta, pupil_mask, rng)
        psf = psf_from_phase(phase, pupil_mask)
        blurred = convolve_fft(truth, psf)
        frames[j] = blurred + noise_sigma * rng.standard_normal(truth.shape)
        phases.append(phase)
        psfs.append(psf)
    return frames, phases, psfs


J = 8
frames, phases, psfs = simulate_frames(
    truth, n_frames=J, n_modes=21, D_over_r0=8.0,
    pupil_mask=pupil_mask, rho=rho, theta=theta,
    noise_sigma=0.01, rng=rng,
)

fig, axes = plt.subplots(2, 4, figsize=(14, 7))
for ax, frame in zip(axes.flat, frames):
    ax.imshow(frame, cmap='gray', vmin=truth.min(), vmax=truth.max())
    ax.axis('off')
plt.suptitle(f'J={J} simulated frames (blurred + noisy) / 시뮬레이션 프레임 {J}개')
plt.tight_layout()
plt.show()

## Part 5: Single-Frame Wiener Deconvolution / 단일 프레임 Wiener 디컨볼루션

**English.** If the PSF is *known* (non-blind case), the Wiener filter gives
$$\hat F = \frac{S^{*}D}{|S|^2 + \gamma_{\rm obj}}.$$
In the truly blind case the PSF is unknown — this is what MFBD–LEC solves jointly. Here we first verify the Wiener filter works when the PSF is known.

**Korean.** PSF가 *알려진* 경우(비-블라인드) Wiener 필터는 위 식을 준다. 진정한 블라인드 경우에는 PSF를 모르며 — 이것이 MFBD–LEC가 공동으로 푸는 문제다. 여기서는 먼저 PSF를 알 때 Wiener 필터가 작동함을 확인한다.

In [ ]:
def wiener_single(
    frame: np.ndarray,
    psf: np.ndarray,
    gamma_obj: float,
) -> np.ndarray:
    """Single-frame Wiener deconvolution with known PSF.

    Args:
        frame: Noisy blurred image d.
        psf: Known PSF h (centred).
        gamma_obj: Object-side regularisation parameter (~1/SNR).

    Returns:
        Real-valued deconvolved image F_hat.
    """
    D = fft2(frame)
    S = fft2(ifftshift(psf))
    F_hat = (np.conj(S) * D) / (np.abs(S) ** 2 + gamma_obj)
    return np.real(ifft2(F_hat))


restored = wiener_single(frames[0], psfs[0], gamma_obj=0.005)

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
axes[0].imshow(truth, cmap='gray'); axes[0].set_title('Truth / 진정'); axes[0].axis('off')
axes[1].imshow(frames[0], cmap='gray'); axes[1].set_title('Noisy blurred / 잡음 + 블러'); axes[1].axis('off')
axes[2].imshow(restored, cmap='gray'); axes[2].set_title('Wiener restored / Wiener 복원'); axes[2].axis('off')
plt.tight_layout(); plt.show()

err_raw = ((frames[0] - truth) ** 2).mean()
err_rec = ((restored - truth) ** 2).mean()
print(f'MSE raw     : {err_raw:.5f}')
print(f'MSE restored: {err_rec:.5f} (known PSF, single frame)')

## Part 6: Multi-Frame Wiener Object Estimate / 다중 프레임 Wiener 물체 추정

**English.** Löfdahl's Eq. (4) gives the maximum-likelihood object estimate across $J$ frames when the OTFs are known:
$$\hat F = \frac{\sum_j S_j^{*} D_j}{\gamma_{\rm obj} + \sum_j |S_j|^2}.$$
Increasing $J$ raises the SNR (noise averaging) and — crucially — fills in the nulls of individual OTFs where different frames have different passbands. This is the *information-theoretic* benefit of MFBD.

**Korean.** Löfdahl 식 (4)는 OTF가 알려졌을 때 $J$개 프레임에 걸친 최대 우도 물체 추정을 준다. $J$를 늘리면 SNR이 높아지고(잡음 평균화) — 결정적으로 — 서로 다른 프레임이 서로 다른 통과 대역을 가질 때 개별 OTF의 영점을 서로 메워준다. 이것이 MFBD의 *정보 이론적* 이득이다.

In [ ]:
def wiener_multi(
    frames: np.ndarray,
    psfs: list[np.ndarray],
    gamma_obj: float,
) -> np.ndarray:
    """Multi-frame Wiener object estimate given known PSFs (Löfdahl Eq. 4).

    Args:
        frames: Array of shape (J, H, W) of observed noisy frames.
        psfs: List of J known PSFs, each centred and unit-sum.
        gamma_obj: Object-side regulariser.

    Returns:
        Real-valued restored object F_hat.
    """
    num = np.zeros(frames[0].shape, dtype=complex)
    den = np.full(frames[0].shape, gamma_obj)
    for d, h in zip(frames, psfs):
        D = fft2(d)
        S = fft2(ifftshift(h))
        num += np.conj(S) * D
        den += np.abs(S) ** 2
    return np.real(ifft2(num / den))


# Compare single-frame vs increasing J multi-frame reconstructions
J_values = [1, 2, 4, 8]
fig, axes = plt.subplots(1, len(J_values) + 1, figsize=(18, 4))
axes[0].imshow(truth, cmap='gray'); axes[0].set_title('Truth'); axes[0].axis('off')
for ax, J_use in zip(axes[1:], J_values):
    rec = wiener_multi(frames[:J_use], psfs[:J_use], gamma_obj=0.005)
    mse = ((rec - truth) ** 2).mean()
    ax.imshow(rec, cmap='gray')
    ax.set_title(f'J={J_use}\nMSE={mse:.5f}')
    ax.axis('off')
plt.tight_layout(); plt.show()

## Part 7: Illustration of the LEC Null-Space Structure / LEC 영공간 구조의 예시

**English.** For Phase Diversity with $K$ channels and $M$ modes with fully known differences, Löfdahl's Eq. (23) gives
$$\mathbf{C}^{\rm PD'} = \begin{pmatrix}\mathbf{I}_M & -\mathbf{I}_M & & \\ \vdots & & \ddots & \\ \mathbf{I}_M & & & -\mathbf{I}_M\end{pmatrix},\qquad \mathbf{Q}_2 = \frac{1}{\sqrt K}\begin{pmatrix}\mathbf{I}_M \\ \vdots \\ \mathbf{I}_M\end{pmatrix}.$$
Below we build $\mathbf{C}$ explicitly, compute $\mathbf{Q}_2$ via QR factorization, and verify $\mathbf{C}\mathbf{Q}_2 = 0$ and $\mathbf{Q}_2^{\top}\mathbf{Q}_2 = \mathbf{I}$.

**Korean.** 완전히 알려진 차이를 가진 $K$ 채널 $M$ 모드 PD에서, Löfdahl 식 (23)은 위 블록 형태를 준다. 아래에서 $\mathbf{C}$를 명시적으로 구성하고, QR 분해로 $\mathbf{Q}_2$를 계산해, $\mathbf{C}\mathbf{Q}_2 = 0$과 $\mathbf{Q}_2^{\top}\mathbf{Q}_2 = \mathbf{I}$를 확인한다.

In [ ]:
def build_pd_constraint(K: int, M: int) -> np.ndarray:
    """Construct the Phase-Diversity constraint matrix C^PD' (fully known diffs).

    Enforces alpha_{1,m} = alpha_{k,m} for all k > 1 and all m.
    The lexicographic ordering puts m varying fastest within each k.

    Args:
        K: Number of diversity channels.
        M: Number of wavefront modes per channel.

    Returns:
        Constraint matrix C of shape ((K-1)*M, K*M).
    """
    rows = (K - 1) * M
    cols = K * M
    C = np.zeros((rows, cols))
    for k in range(1, K):
        row_start = (k - 1) * M
        C[row_start:row_start + M, 0:M] = np.eye(M)
        C[row_start:row_start + M, k * M:(k + 1) * M] = -np.eye(M)
    return C


def null_space_from_qr(C: np.ndarray) -> np.ndarray:
    """Compute an orthonormal basis of the null space of C via QR of C^T.

    Args:
        C: Constraint matrix of shape (N_C, N).

    Returns:
        Q2 of shape (N, N - N_C) with Q2.T @ Q2 = I and C @ Q2 = 0.
    """
    N_C, N = C.shape
    Q, _ = np.linalg.qr(C.T, mode='complete')
    Q2 = Q[:, N_C:]
    return Q2


K, M = 4, 12
C = build_pd_constraint(K, M)
Q2 = null_space_from_qr(C)

print(f'C shape       : {C.shape}')
print(f'Q2 shape      : {Q2.shape}')
print(f'max|C @ Q2|   : {np.max(np.abs(C @ Q2)):.2e}  (should be ~0)')
print(f'max|Q2^T Q2 - I|: {np.max(np.abs(Q2.T @ Q2 - np.eye(Q2.shape[1]))):.2e}  (should be ~0)')

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].imshow(C, cmap='RdBu_r', vmin=-1, vmax=1)
axes[0].set_title(f'C^PD\'  ({C.shape[0]} x {C.shape[1]})')
axes[1].imshow(Q2, cmap='RdBu_r')
axes[1].set_title(f'Q_2  ({Q2.shape[0]} x {Q2.shape[1]})')
plt.tight_layout(); plt.show()

## Summary / 요약

| Concept / 개념 | This Paper / 이 논문 | Modern Equivalent / 현대 동등물 |
|---|---|---|
| Basis for phase / 위상 기저 | Karhunen–Loève (KL) modes, ~15 modes | Same; MOMFBD uses 30–50 KL modes |
| Wavefront prior / 파면 사전분포 | Tikhonov $\gamma_{\rm wf}/\lambda_m$ | Same, plus sparsity priors and Bayesian extensions |
| Constraint handling / 제약 처리 | Null-space QR of $\mathbf{C}^{\top}$ | Same; augmented Lagrangian for soft constraints |
| Object estimate / 물체 추정 | Wiener filter Eq. (4) | Same in MOMFBD; Bayesian variants add hyperpriors |
| Parallelism / 병렬화 | Block-diagonal MFBD Hessian | Distributed across sub-fields and GPUs (DKIST pipeline) |
| Extensibility / 확장성 | Add rows to $\mathbf{C}$ for multi-object, dual $\lambda$, SH+imager | Configured via config files in MOMFBD; new observing modes (SPINOR, ViSP, VBI) slot in naturally |

**English.** The MFBD–LEC formulation is the theoretical seed of MOMFBD (van Noort, Rouppe van der Voort, Löfdahl 2005), the de-facto post-processing pipeline at the Swedish Solar Telescope and GREGOR, and the template now being adapted for DKIST's Visible Broadband Imager. Our notebook exercised the mechanical building blocks; the full MFBD–LEC optimization (Newton iterations on $\boldsymbol\beta$ with conjugate-gradient solves of the reduced Hessian system) is beyond the scope of this demo but would follow directly from the pieces above.

**Korean.** MFBD–LEC 수식화는 MOMFBD(van Noort, Rouppe van der Voort, Löfdahl 2005)의 이론적 씨앗이며, 스웨덴 태양 망원경(SST)과 GREGOR의 사실상의 후처리 파이프라인이고, 현재 DKIST의 가시광 광대역 영상기를 위해 적용되고 있는 템플릿이다. 이 노트북은 기계적 구성 요소를 연습했으며, 완전한 MFBD–LEC 최적화($\boldsymbol\beta$에 대한 뉴턴 반복과 축소 헤시안 시스템의 공액 경사 해)는 이 데모 범위를 벗어나지만 위 조각들로부터 직접 이어진다.